导入标准模块:


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    from IPython.display import HTML, display
except ImportError:
    HTML = None
    display = None

STYLE_PATH = Path("../style/course.css")
TOGGLE_PATH = Path("../style/code_toggle.html")

if HTML is not None and display is not None:
    if STYLE_PATH.exists():
        display(HTML(f"<style>{STYLE_PATH.read_text(encoding='utf-8')}</style>"))
    if TOGGLE_PATH.exists():
        display(HTML(TOGGLE_PATH.read_text(encoding="utf-8")))

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.grid"] = True


本节不再依赖旧版 `data/analogue_spectrum.npz` 之类的外部中间文件，而是直接在 notebook 内合成电压流、噪声和几何延迟。这样做有两个好处：

1. 我们可以把“采样、量化、通道化、延迟补偿、相关、数据落盘”看成一条连续的数字链路；
2. 每个图和每个数值都能由学生亲自重跑并修改，从而真正建立对相关器工作方式的直觉。

本节和上一节 [7.3 模拟电子学](7_3_analogue.ipynb) 对接：上一节讨论的是射频到中频、系统温度和模拟增益，本节则讨论 ADC 之后的数字系统如何把离散电压流变成可见度。


***


## 7.4 数字相关器：采样、通道化与相关

对于干涉仪而言，数字系统至少要完成以下几件事：

- 以足够高的采样率记录各天线的电压流；
- 在可接受的数据率下完成量化与重采样；
- 把宽带时域数据分解为窄带频率通道；
- 对每条基线、每个极化、每个频率通道形成相关积；
- 附上时间、频率、天线编号、`uvw`、权重和标记，写成后端数据产品。

用符号表示时，FX 相关器的核心结果可以写成

$$
V_{pq}(\nu) = \left\langle X_p(\nu) X_q^*(\nu) \right\rangle,
$$

其中 $X_p(\nu)$ 是第 $p$ 面天线在某个时间块经过通道化后的复频谱。这里的角括号表示时间平均或积累。


In [ ]:
def make_real_if_signal(sample_rate_hz, duration_s, tones_hz, amplitudes, phases=None, noise_std=0.0, rng=None):
    t = np.arange(0.0, duration_s, 1.0 / sample_rate_hz)
    signal = np.zeros_like(t)
    phases = np.zeros(len(tones_hz)) if phases is None else np.asarray(phases)
    for freq, amp, phase in zip(tones_hz, amplitudes, phases):
        signal += amp * np.cos(2.0 * np.pi * freq * t + phase)
    if noise_std > 0.0:
        rng = np.random.default_rng(0) if rng is None else rng
        signal += rng.normal(scale=noise_std, size=t.size)
    return t, signal


def real_fft_spectrum(signal, sample_rate_hz, window="hann"):
    x = np.asarray(signal)
    if window == "hann":
        weights = np.hanning(x.size)
    else:
        weights = np.ones(x.size)
    spectrum = np.fft.rfft(x * weights)
    freqs = np.fft.rfftfreq(x.size, d=1.0 / sample_rate_hz)
    amplitude = np.abs(spectrum) / np.sum(weights)
    return freqs, amplitude


def quantize_real(x, nbits, full_scale=1.0):
    levels = 2 ** nbits
    step = 2.0 * full_scale / levels
    clipped = np.clip(x, -full_scale, full_scale - step)
    indices = np.floor((clipped + full_scale) / step)
    return -full_scale + (indices + 0.5) * step


def fft_channelize(signal, sample_rate_hz, nfft=1024, window="hann"):
    nblock = len(signal) // nfft
    blocks = np.asarray(signal[: nblock * nfft]).reshape(nblock, nfft)
    if window == "hann":
        weights = np.hanning(nfft)
    else:
        weights = np.ones(nfft)
    spectra = np.fft.rfft(blocks * weights, axis=1)
    power = np.mean(np.abs(spectra) ** 2, axis=0)
    freqs = np.fft.rfftfreq(nfft, d=1.0 / sample_rate_hz)
    return freqs, power


def bandlimited_complex_noise(n, sample_rate_hz, f_low_hz, f_high_hz, rng):
    freqs = np.fft.fftfreq(n, d=1.0 / sample_rate_hz)
    spectrum = np.zeros(n, dtype=complex)
    band = (np.abs(freqs) >= f_low_hz) & (np.abs(freqs) <= f_high_hz)
    spectrum[band] = rng.normal(size=band.sum()) + 1j * rng.normal(size=band.sum())
    signal = np.fft.ifft(spectrum)
    return signal / np.std(signal)


def fractional_delay_complex(signal, tau_s, sample_rate_hz):
    freqs = np.fft.fftfreq(signal.size, d=1.0 / sample_rate_hz)
    spectrum = np.fft.fft(signal)
    delayed = spectrum * np.exp(-2j * np.pi * freqs * tau_s)
    return np.fft.ifft(delayed)


def fx_cross_spectrum(x, y, sample_rate_hz, nfft=1024, window="hann"):
    nblock = min(len(x), len(y)) // nfft
    x_blocks = np.asarray(x[: nblock * nfft]).reshape(nblock, nfft)
    y_blocks = np.asarray(y[: nblock * nfft]).reshape(nblock, nfft)
    if window == "hann":
        weights = np.hanning(nfft)
    else:
        weights = np.ones(nfft)
    X = np.fft.fftshift(np.fft.fft(x_blocks * weights, axis=1), axes=1)
    Y = np.fft.fftshift(np.fft.fft(y_blocks * weights, axis=1), axes=1)
    vis = np.mean(X * np.conj(Y), axis=0)
    freqs = np.fft.fftshift(np.fft.fftfreq(nfft, d=1.0 / sample_rate_hz))
    return freqs, vis


def circular_lag_correlator(x, y):
    n = len(x)
    return np.array([np.mean(np.roll(x, -lag) * np.conj(y)) for lag in range(n)])


def quantize_2bit_optimal(x, threshold=0.9816):
    q = np.empty_like(x)
    q[x < -threshold] = -3.0
    q[(x >= -threshold) & (x < 0.0)] = -1.0
    q[(x >= 0.0) & (x < threshold)] = 1.0
    q[x >= threshold] = 3.0
    return q / np.sqrt(np.mean(q**2))


def baseline_pairs(nant):
    ant1, ant2 = np.triu_indices(nant, k=1)
    return list(zip(ant1.tolist(), ant2.tolist()))


### 7.4.1 ADC、奈奎斯特采样与量化

数字系统的第一步是把模拟中频（或基带）电压变成离散采样序列。对于**实数采样**，若信号最高频率为 $\nu_{\max}$，则采样率至少要满足

$$
\nu_s > 2\nu_{\max}.
$$

否则高频分量会折叠到低频，产生混叠（aliasing）。此外，采样后的数值还要经过有限比特量化。量化比特数越少，数据率越低，但量化误差和后续相关损失也越大。


In [ ]:
rng = np.random.default_rng(20260418)

tones_hz = [8e6, 18e6]
amplitudes = [1.0, 0.65]
phases = [0.15, 1.05]
duration_s = 80e-6
sample_rates = [64e6, 30e6]

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
peak_summary = []

for row, fs in enumerate(sample_rates):
    t, signal = make_real_if_signal(fs, duration_s, tones_hz, amplitudes, phases=phases, noise_std=0.05, rng=rng)
    signal = signal / np.max(np.abs(signal))

    nshow = min(320, signal.size)
    axes[row, 0].plot(t[:nshow] * 1e6, signal[:nshow], lw=1.0)
    axes[row, 0].set_title(f"sampling at {fs / 1e6:.0f} MHz")
    axes[row, 0].set_xlabel("time [us]")
    axes[row, 0].set_ylabel("normalized voltage")

    freqs, amp = real_fft_spectrum(signal, fs)
    axes[row, 1].plot(freqs / 1e6, amp, lw=1.2)
    axes[row, 1].set_xlim(0.0, fs / 2e6)
    axes[row, 1].set_xlabel("frequency [MHz]")
    axes[row, 1].set_ylabel("amplitude")

    peak_freqs = freqs[np.argsort(amp)[-4:]] / 1e6
    peak_summary.append((fs / 1e6, np.sort(peak_freqs)))

fig.suptitle("Same IF signal under different sampling rates", fontsize=14)
fig.tight_layout()
plt.show()
plt.close(fig)

_, reference = make_real_if_signal(
    64e6,
    40e-6,
    tones_hz,
    amplitudes,
    phases=phases,
    noise_std=0.03,
    rng=np.random.default_rng(7),
)
reference = reference / (1.15 * np.max(np.abs(reference)))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(reference[:120], label="float", lw=1.5, color="black")

quantization_errors = {}
for nbits, color in zip([2, 4, 8], ["tab:red", "tab:blue", "tab:green"]):
    quantized = quantize_real(reference, nbits)
    quantization_errors[nbits] = np.sqrt(np.mean((reference - quantized) ** 2))
    ax.step(np.arange(120), quantized[:120], where="mid", label=f"{nbits}-bit", alpha=0.8, color=color)

ax.set_title("Quantization bit depth versus waveform fidelity")
ax.set_xlabel("sample index")
ax.set_ylabel("normalized voltage")
ax.legend()
fig.tight_layout()
plt.show()
plt.close(fig)

print("频谱峰值位置：")
for fs_mhz, peaks_mhz in peak_summary:
    print(f"  sampling = {fs_mhz:5.1f} MHz -> peaks ≈ {np.array2string(peaks_mhz, precision=3)} MHz")

print("\n量化误差（RMS）：")
for nbits in [2, 4, 8]:
    print(f"  {nbits}-bit -> rms error = {quantization_errors[nbits]:.5f}")


这个实验说明了三件事：

- 当采样率为 `64 MHz` 时，`8 MHz` 和 `18 MHz` 两个分量都能被正确恢复；
- 当采样率降到 `30 MHz` 时，`18 MHz` 已经超过奈奎斯特频率 `15 MHz`，因此会折叠为 `12 MHz`；
- 在相同满量程下，`2-bit` 量化的波形台阶非常明显，而 `8-bit` 已经足以逼近连续电压。

在真实系统里，ADC 前通常还需要模拟抗混叠滤波器，否则“采了再说”往往意味着把不可逆的频谱折叠直接写进数据。


### 7.4.2 通道化：FFT、窗函数与 PFB 的动机

ADC 输出的只是时域样本。相关器要想形成**按频率分开的可见度**，必须先做通道化（channelization）。最直接的方法是把长度为 $N$ 的时间块做 FFT，得到离散频率通道。

但一个重要细节是：如果信号频率没有正好落在某个 FFT 栅格上，能量会泄漏到邻近通道。工程上广泛使用的多相滤波器组（polyphase filter bank, PFB）本质上就是为了减弱这种谱泄漏和通道串扰。下面先用一个“加窗 FFT”的实验来理解它的核心动机。


In [ ]:
fs = 32e6
nfft = 1024
t = np.arange(nfft) / fs
tone_freq = 3.37e6
tone = np.cos(2.0 * np.pi * tone_freq * t)

freqs_rect, amp_rect = real_fft_spectrum(tone, fs, window="rect")
freqs_hann, amp_hann = real_fft_spectrum(tone, fs, window="hann")

rect_db = 20.0 * np.log10(amp_rect / np.max(amp_rect) + 1e-12)
hann_db = 20.0 * np.log10(amp_hann / np.max(amp_hann) + 1e-12)

rect_peak = np.argmax(amp_rect)
hann_peak = np.argmax(amp_hann)
rect_mask = np.ones_like(rect_db, dtype=bool)
hann_mask = np.ones_like(hann_db, dtype=bool)
rect_mask[max(0, rect_peak - 2) : min(rect_db.size, rect_peak + 3)] = False
hann_mask[max(0, hann_peak - 2) : min(hann_db.size, hann_peak + 3)] = False

max_sidelobe_rect = rect_db[rect_mask].max()
max_sidelobe_hann = hann_db[hann_mask].max()

_, wideband = make_real_if_signal(
    fs,
    3.2e-3,
    [1.25e6, 4.75e6, 7.125e6],
    [1.0, 0.65, 0.4],
    phases=[0.0, 0.8, -0.6],
    noise_std=0.15,
    rng=np.random.default_rng(1234),
)
chan_freqs, chan_power = fft_channelize(wideband, fs, nfft=nfft, window="hann")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(freqs_rect / 1e6, rect_db, label="rectangular")
axes[0].plot(freqs_hann / 1e6, hann_db, label="hann")
axes[0].set_xlim(0.0, 8.0)
axes[0].set_ylim(-90.0, 3.0)
axes[0].set_xlabel("frequency [MHz]")
axes[0].set_ylabel("relative level [dB]")
axes[0].set_title("Window choice changes spectral leakage")
axes[0].legend()

axes[1].plot(
    chan_freqs / 1e6,
    10.0 * np.log10(chan_power / np.max(chan_power) + 1e-12),
    color="tab:purple",
)
axes[1].set_xlim(0.0, 10.0)
axes[1].set_ylim(-70.0, 3.0)
axes[1].set_xlabel("channel center [MHz]")
axes[1].set_ylabel("relative power [dB]")
axes[1].set_title("Channelized spectrum after block averaging")

fig.tight_layout()
plt.show()
plt.close(fig)

channel_width_khz = (chan_freqs[1] - chan_freqs[0]) / 1e3
print(f"FFT channel width = {channel_width_khz:.2f} kHz")
print(f"rectangular window: strongest sidelobe ≈ {max_sidelobe_rect:.2f} dB")
print(f"hann window       : strongest sidelobe ≈ {max_sidelobe_hann:.2f} dB")


从这里可以看到：

- 矩形窗保留了最窄的主瓣，但旁瓣高，容易把强源能量泄到邻近通道；
- `Hann` 窗显著压低旁瓣，代价是主瓣略宽、通道间独立性略差；
- 真正的 PFB 比“单次加窗 FFT”更进一步，它通过多抽头滤波和抽取，让每个输出通道的频响更接近理想带通。

这也是为什么现代宽带阵列几乎都会认真设计 F-engine，而不会把“直接 FFT”视为一个可忽略的工程细节。


### 7.4.3 延迟补偿、相位旋转与 FX 可见度

设第 $q$ 面天线相对于第 $p$ 面天线多走了一段几何延迟 $\tau_g$。在频域里，这会表现为线性相位斜率：

$$
X_q(\nu) = X_p(\nu) e^{-2\pi i \nu \tau_g}.
$$

因此若我们直接做相关，得到的可见度相位会随频率线性变化。数字后端通常会用延迟模型先去掉这条相位斜率，再做积分平均。下面用一个带限复电压流来模拟这个过程。


In [ ]:
rng = np.random.default_rng(123)

fs = 32e6
n_samples = 32768
f_low_hz, f_high_hz = 0.6e6, 7.5e6

x = bandlimited_complex_noise(n_samples, fs, f_low_hz, f_high_hz, rng)

tau_geom = 137e-9
phi_instr = 0.7
y = 0.92 * fractional_delay_complex(x, tau_geom, fs) * np.exp(1j * phi_instr)
y += 0.12 * (rng.normal(size=n_samples) + 1j * rng.normal(size=n_samples))

freqs, vis_raw = fx_cross_spectrum(x, y, fs, nfft=1024, window="hann")

occupied = (
    (np.abs(freqs) >= f_low_hz)
    & (np.abs(freqs) <= f_high_hz)
    & (np.abs(vis_raw) > 0.05 * np.max(np.abs(vis_raw)))
)

phase_raw = np.unwrap(np.angle(vis_raw[occupied]))
coef_raw = np.polyfit(freqs[occupied], phase_raw, 1)
tau_est = coef_raw[0] / (2.0 * np.pi)

vis_corr = vis_raw * np.exp(-2j * np.pi * freqs * tau_geom) * np.exp(1j * phi_instr)
phase_corr = np.unwrap(np.angle(vis_corr[occupied]))
coef_corr = np.polyfit(freqs[occupied], phase_corr, 1)

residual_delay_ps = coef_corr[0] / (2.0 * np.pi) * 1e12
mean_residual_phase_deg = np.degrees(np.angle(np.mean(vis_corr[occupied])))

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

axes[0].plot(freqs / 1e6, np.abs(vis_raw), label="raw visibility")
axes[0].plot(freqs / 1e6, np.abs(vis_corr), label="after delay correction", alpha=0.8)
axes[0].set_ylabel("|Vpq|")
axes[0].set_title("FX visibility amplitude")
axes[0].legend()

axes[1].plot(freqs[occupied] / 1e6, np.unwrap(np.angle(vis_raw[occupied])), label="raw phase")
axes[1].plot(freqs[occupied] / 1e6, np.unwrap(np.angle(vis_corr[occupied])), label="after correction")
axes[1].set_xlabel("frequency [MHz]")
axes[1].set_ylabel("phase [rad]")
axes[1].set_title("Phase slope before and after delay correction")
axes[1].legend()

fig.tight_layout()
plt.show()
plt.close(fig)

print(f"input geometric delay      = {tau_geom * 1e9:.2f} ns")
print(f"estimated from phase slope = {tau_est * 1e9:.2f} ns")
print(f"residual delay after correction = {residual_delay_ps:.2f} ps")
print(f"mean residual phase            = {mean_residual_phase_deg:.3f} deg")


这正是 FX 相关器里“延迟补偿 + 条纹跟踪（fringe rotation）”的核心思想。对真实阵列来说，$\tau_g$ 不是常数，而是随源位置、基线、地球自转和相位中心选择不断变化的，因此后端必须持续更新模型。

需要注意的是：

- **整数采样级**的粗延迟，通常在时域里直接移动样本；
- **分数采样级**的精细延迟，通常在频域里用线性相位因子实现；
- 对很长时间平均而言，如果不做这些补偿，带宽和时间平均都会导致条纹被洗掉。


### 7.4.4 XF 等价性与重采样代价

历史上并不是所有相关器都先做 FFT。早期系统中常见的是 XF 或 lag correlator：先计算一组时滞相关，再把相关函数做傅里叶变换得到谱域可见度。只要定义一致，FX 与 XF 的输出在数学上应当等价。

另一方面，工程上常常会在 F-engine 后再次降比特数，以节省 corner turn 和 X-engine 的链路带宽。这会带来相关效率损失，需要 Van Vleck 一类修正或离线标定。


In [ ]:
nfft_block = 512
x_block = x[:nfft_block]
y_block = y[:nfft_block]

fx_block = np.fft.fft(x_block) * np.conj(np.fft.fft(y_block))
xf_block = np.fft.fft(circular_lag_correlator(x_block, y_block)) * nfft_block
fx_xf_max_diff = np.max(np.abs(fx_block - xf_block))

rng = np.random.default_rng(5)
rho = 0.35
samples = rng.multivariate_normal([0.0, 0.0], [[1.0, rho], [rho, 1.0]], size=250000)
a, b = samples.T

corr_true = np.mean(a * b)
sign_a = np.where(a >= 0.0, 1.0, -1.0)
sign_b = np.where(b >= 0.0, 1.0, -1.0)
corr_1bit_raw = np.mean(sign_a * sign_b)
corr_1bit_recovered = np.sin(0.5 * np.pi * corr_1bit_raw)

q2_a = quantize_2bit_optimal(a)
q2_b = quantize_2bit_optimal(b)
corr_2bit_raw = np.mean(q2_a * q2_b)

fig, ax = plt.subplots(figsize=(8, 4))
labels = ["float", "1-bit raw", "1-bit corrected", "2-bit raw"]
values = [corr_true, corr_1bit_raw, corr_1bit_recovered, corr_2bit_raw]
colors = ["black", "tab:red", "tab:green", "tab:blue"]
ax.bar(labels, values, color=colors, alpha=0.85)
ax.set_ylabel("estimated correlation coefficient")
ax.set_title("Quantization lowers correlation accuracy; correction helps")
fig.tight_layout()
plt.show()
plt.close(fig)

print(f"max |FX - XF| for one block = {fx_xf_max_diff:.3e}")
print(f"true correlation             = {corr_true:.5f}")
print(f"1-bit raw estimate           = {corr_1bit_raw:.5f}")
print(f"1-bit Van Vleck recovery     = {corr_1bit_recovered:.5f}")
print(f"2-bit raw estimate           = {corr_2bit_raw:.5f}")


这里要抓住两个工程事实：

- **FX/XF 的差别首先是实现路径，而不是最终物理量**。如果时滞定义、窗函数、积分方式都一致，最后得到的谱域相关量应相同；
- **重采样并不是“免费压缩”**。即使相关器还能工作，量化也会改变量值统计特性，所以必须用已知的校正关系或实验标定把偏差补回来。


### 7.4.5 Corner Turn、相关矩阵与算量扩展

完成 F-engine 之后，数据通常还是按“天线-时间”顺序组织的；而 X-engine 更希望按“频率-时间”顺序拿到所有天线在同一通道上的数据。这一步全局转置就是 **corner turn**。它本身不改变物理信息，却常常决定整个系统的交换网络、内存带宽和可扩展性。


In [ ]:
nant = 7
corr_matrix = np.zeros((nant, nant))
corr_matrix[np.diag_indices(nant)] = 2.0
upper = np.triu_indices(nant, k=1)
corr_matrix[upper] = 1.0
corr_matrix[(upper[1], upper[0])] = 1.0

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

im = axes[0].imshow(corr_matrix, cmap="Blues", vmin=0.0, vmax=2.0)
axes[0].set_title("Correlation matrix for a 7-antenna array")
axes[0].set_xlabel("antenna q")
axes[0].set_ylabel("antenna p")
axes[0].set_xticks(range(nant))
axes[0].set_yticks(range(nant))

nant_counts = np.arange(2, 65)
baseline_counts = nant_counts * (nant_counts - 1) // 2
axes[1].plot(nant_counts, baseline_counts, color="tab:orange", lw=2.0)
axes[1].set_title("Cross-baseline count versus number of antennas")
axes[1].set_xlabel("number of antennas")
axes[1].set_ylabel("cross baselines")

fig.tight_layout()
plt.show()
plt.close(fig)

ntime, nants, npol, nchan = 8, 16, 2, 1024
voltage_cube = np.zeros((ntime, nants, npol, nchan), dtype=np.int8)
corner_cube = np.transpose(voltage_cube, (3, 0, 1, 2))

nchan_fft = 4096
nant_scale = np.array([16, 64, 128, 256])
f_engine_cost = nant_scale * nchan_fft * np.log2(nchan_fft)
x_engine_cost = (nant_scale * (nant_scale + 1) // 2) * nchan_fft
x_share = x_engine_cost / (f_engine_cost + x_engine_cost)

print(f"for {nant} antennas: cross baselines = {nant * (nant - 1) // 2}, including autos = {nant * (nant + 1) // 2}")
print(f"corner turn: {voltage_cube.shape} -> {corner_cube.shape}")
print("\nApproximate per-block cost split in an FX correlator:")
for n_ant, share in zip(nant_scale, x_share):
    print(f"  Nant = {n_ant:3d} -> X-engine share ≈ {100.0 * share:5.1f}%")


这个例子反映了现代相关器设计中的一个常见判断：

- 小阵列时，F-engine 和 X-engine 都不算太重；
- 天线数一大，相关矩阵大小按 $N_{\rm ant}^2$ 增长，X-engine 与数据交换马上成为瓶颈；
- 因此“能不能做完 FFT”通常不是最难的问题，“如何把所有天线的同一频点高效送到一起”才是硬件系统设计的关键。


### 7.4.6 数据产品：Measurement Set 的最小骨架

相关器并不会只输出一个复数矩阵。后续标定和成像需要知道每条可见度对应的：

- 哪两面天线；
- 观测时间和积分长度；
- `uvw` 坐标；
- 频率通道与极化；
- 权重与数据标记（flags）。

业界常用的数据容器是 Measurement Set（MS）。下面不去重现完整标准，而是构造一个“最小 MS 骨架”，帮助建立表结构直觉。


In [ ]:
rng = np.random.default_rng(11)

nant_ms = 5
pairs = baseline_pairs(nant_ms)
ntime = 4
nchan = 8
npol = 4
nrow = len(pairs) * ntime

antenna1 = np.empty(nrow, dtype=np.int32)
antenna2 = np.empty(nrow, dtype=np.int32)
time_col = np.empty(nrow, dtype=float)
uvw = np.empty((nrow, 3), dtype=float)
data = np.empty((nrow, nchan, npol), dtype=np.complex128)
flag = np.zeros((nrow, nchan, npol), dtype=bool)
weight = np.empty((nrow, npol), dtype=float)

row = 0
for time_index in range(ntime):
    for a1, a2 in pairs:
        antenna1[row] = a1
        antenna2[row] = a2
        time_col[row] = 60000.0 + 10.0 * time_index
        uvw[row] = [12.0 * (a2 - a1), 5.0 * time_index, 0.3 * (a1 + a2)]

        noise = 0.05 * (rng.normal(size=(nchan, npol)) + 1j * rng.normal(size=(nchan, npol)))
        phase = np.exp(-2j * np.pi * np.linspace(-0.2, 0.2, nchan))[:, None]
        data[row] = phase * (1.0 + noise)

        flag[row, -1, 2:] = time_index == 2
        weight[row] = np.array([1.0, 1.0, 0.8, 0.8])
        row += 1

print(f"rows in MAIN table = {nrow}")
print(f"DATA   shape = {data.shape}")
print(f"UVW    shape = {uvw.shape}")
print(f"FLAG   shape = {flag.shape}")
print(f"WEIGHT shape = {weight.shape}")

print("\n前三行示例：")
for i in range(3):
    amp00 = np.abs(data[i, 0, 0])
    print(
        f"row {i:02d}: ant=({antenna1[i]},{antenna2[i]}), "
        f"time={time_col[i]:.1f}, uvw={uvw[i]}, "
        f"|DATA[0,0]|={amp00:.3f}, flagged={flag[i, -1, 2]}, weight_xx={weight[i, 0]:.1f}"
    )


到这里，我们已经把数字后端的主线完整串起来了：

- 先用采样和量化把模拟电压变成可处理的数据流；
- 再通过通道化形成窄带频域表示；
- 用延迟模型和相位旋转保证相关不被条纹洗掉；
- 在 X-engine 中形成全阵列可见度；
- 最后连同 `uvw`、权重和标记一起写入观测数据产品。

后续几节会继续讨论这些数字可见度如何受到主波束、偏振、传播效应和 RFI 的影响。到那时，学生应当把本节视为一个“工程接口层”：它把天空电磁场和后续校准/成像之间，用一套可执行的数字流程连接了起来。
